# Week 4 &mdash; 3D Data and Point Clouds

## Theory: Deep Learning on 3D Objects, Rotation Equivariance, and PointNet

**Geometric Deep Learning &mdash; From Foundations to Applications**

---

### Learning Objectives

1. Describe the principal representations of 3D data and their symmetry challenges.
2. Characterise the **permutation** and **rigid-motion ($SE(3)$)** symmetries of point clouds.
3. Derive the **PointNet** architecture and prove its permutation invariance.
4. Distinguish **invariant** from **equivariant** approaches to rotation.
5. Sketch the principle behind genuinely **$SO(3)$-equivariant** networks (Tensor Field Networks).

---


## 1. Representations of 3D Data

Three-dimensional geometry admits several representations, each with distinct symmetry properties:

- **Voxel grids** discretise space into a regular $\mathbb{Z}^3$ lattice. They inherit the grid structure that makes 3D CNNs applicable, but memory grows cubically with resolution and the regular grid breaks rotational symmetry.
- **Multi-view images** render the object from several cameras and apply 2D CNNs. Effective in practice but indirect, and the choice of views is arbitrary.
- **Meshes** $(V, E, F)$ represent surfaces with vertices, edges, and faces; they carry rich connectivity but irregular structure (treated in Weeks 5&ndash;6).
- **Point clouds** are *unordered sets* of coordinates $\{p_i\}_{i=1}^N \subset \mathbb{R}^3$, possibly with features. They are the raw output of LiDAR and depth sensors and the focus of this week.

A point cloud is the canonical example of an **unordered set embedded in physical space**, so it carries *two* symmetries simultaneously, discussed next.


## 2. The Two Symmetries of a Point Cloud

**Permutation symmetry.** A point cloud is a *set*: the labelling of points is arbitrary. A function $f$ of the cloud must be invariant (for global predictions) or equivariant (for per-point predictions) under any permutation $\pi \in S_N$:
$$
f\big(\{p_{\pi(1)}, \dots, p_{\pi(N)}\}\big) = f\big(\{p_1, \dots, p_N\}\big).
$$

**Rigid-motion symmetry.** The object's identity is unchanged by rotating or translating it in space. The relevant group is $SE(3) = SO(3) \ltimes \mathbb{R}^3$ from Week 1, acting by $p_i \mapsto Rp_i + t$. A classifier should be **$SE(3)$-invariant**; a segmenter or normal-estimator should be **$SE(3)$-equivariant**.

Designing for these two symmetries jointly is the central problem of 3D deep learning. PointNet solves the permutation problem exactly and the rotation problem approximately; equivariant networks address rotation by construction.


## 3. PointNet: Architecture and the Universal Set Function

PointNet (Qi et al., 2017) is the foundational deep architecture for raw point clouds. Its design follows directly from the permutation-invariance requirement.

The core insight is a structural theorem: any continuous permutation-invariant function $f$ on a set can be approximated arbitrarily well in the form
$$
f(\{p_1, \dots, p_N\}) \approx \gamma\!\Big(\underset{i=1,\dots,N}{\operatorname{MAX}}\; h(p_i)\Big),
$$
where $h$ is a shared per-point MLP, $\operatorname{MAX}$ is an element-wise **symmetric** (permutation-invariant) pooling operator, and $\gamma$ is a final MLP. This is precisely the Deep Sets factorisation we met in Week 1, applied to 3D coordinates.

**PointNet pipeline.**

1. **Shared MLP** $h : \mathbb{R}^3 \to \mathbb{R}^K$ embeds each point independently (so permutation-equivariant).
2. **Symmetric pooling** $\max_i h(p_i)$ aggregates into a single $K$-dimensional global feature (permutation-invariant).
3. **Classification head** $\gamma$ maps the global feature to class scores; for **segmentation**, the global feature is concatenated back onto each point's local feature and a second shared MLP predicts per-point labels (permutation-equivariant).

**Proposition.** *The PointNet classification map is permutation-invariant.* Each point is processed identically by $h$, and $\max$ is invariant to reordering its arguments; therefore reordering the input cannot change the pooled feature, and hence cannot change the output. $\;\blacksquare$


## 4. Handling Rotation: Invariance vs. Equivariance

PointNet is permutation-invariant *by construction* but is **not** rotation-invariant: rotating the cloud changes the coordinates fed to $h$, and hence the output. Three families of remedies exist.

**(a) Data augmentation.** Train on many random rotations. Simple and widely used, but gives only *approximate*, statistical invariance &mdash; no guarantee on unseen orientations.

**(b) Input alignment (T-Net).** PointNet includes a small *transformation network* that predicts a $3\times 3$ matrix to canonically align the input. This learns a soft alignment but is neither exactly invariant nor theoretically equivariant, and adds parameters.

**(c) Invariant features.** Replace raw coordinates with rotation-invariant quantities &mdash; pairwise distances $\|p_i - p_j\|$, angles, or other $SE(3)$-invariants. This guarantees invariance but discards directional information that may be useful for the task.

The deepest solution &mdash; preserving directional information *and* respecting rotation &mdash; is genuine **equivariance**, discussed next.


## 5. Toward $SO(3)$-Equivariance: Tensor Field Networks

An ideal 3D network maintains features that *transform correctly* under rotation throughout the network, achieving invariance only at a final read-out &mdash; the Week 1 blueprint applied to $SO(3)$.

**Tensor Field Networks** (Thomas et al., 2018) achieve exact $SE(3)$-equivariance by representing features as **steerable** quantities indexed by representation type $\ell = 0, 1, 2, \dots$ (scalars, vectors, rank-2 tensors, $\dots$), corresponding to the irreducible representations of $SO(3)$. The convolution filters are constrained to be products of learnable radial functions and **spherical harmonics** $Y_\ell^m$, and feature interactions are combined via **Clebsch&ndash;Gordan** coefficients, which precisely encode how angular-momentum representations combine.

The upshot: a type-$\ell$ feature transforms under rotation $R$ by the corresponding **Wigner-D matrix** $D^\ell(R)$, so the network output rotates consistently with its input. The same principle underlies modern equivariant models such as **SE(3)-Transformers** and **e3nn**, now standard in molecular and structural-biology applications (e.g. protein-structure prediction).

The mathematical apparatus &mdash; tangent spaces, intrinsic operators, and the Laplace&ndash;Beltrami operator &mdash; that makes these constructions rigorous on curved domains is developed in Week 5.


## 6. Summary

- 3D data has several representations; **point clouds** are unordered sets in physical space.
- They carry two symmetries: **permutation** ($S_N$) and **rigid motion** ($SE(3)$).
- **PointNet** = shared per-point MLP + **symmetric (max) pooling** + head; it is permutation-invariant by the Deep Sets theorem.
- PointNet is *not* rotation-invariant; remedies are augmentation, T-Net alignment, or invariant features.
- **Tensor Field Networks** and successors achieve *exact* $SE(3)$-equivariance using spherical harmonics and Clebsch&ndash;Gordan tensor products.

### References

- Qi, C. R., Su, H., Mo, K., &amp; Guibas, L. J. (2017). *PointNet: Deep Learning on Point Sets for 3D Classification and Segmentation.* CVPR.
- Zaheer, M., et al. (2017). *Deep Sets.* NeurIPS.
- Thomas, N., et al. (2018). *Tensor Field Networks: Rotation- and Translation-Equivariant Neural Networks for 3D Point Clouds.* arXiv:1802.08219.
- Fuchs, F., et al. (2020). *SE(3)-Transformers.* NeurIPS.

### Exercises

1. Prove that replacing $\max$ with $\mathrm{sum}$ or $\mathrm{mean}$ preserves permutation invariance. Which is most expressive, and why (recall Week 3, GIN)?
2. Show that pairwise distances $\|p_i - p_j\|$ are $SE(3)$-invariant.
3. Explain why a vector ($\ell=1$) feature must transform as $v \mapsto Rv$ under rotation for the network to be equivariant.
